# Cricket No-Ball Detection: Vision Pipeline & Rule Engine Walkthrough

This notebook demonstrates the end-to-end evaluation pipeline for cricket delivery adjudication:
1. **Front-Foot Popping Crease Model (MCC Law 21.5):** 4-stage Residual SE-CNN evaluating side-on foot landing.
2. **Waist-High Full-Toss Model (MCC Law 41.7):** 4-stage Residual SE-CNN evaluating ball height at the batter's plane.
3. **Spatial Attribution (Grad-CAM):** Stage-4 feature activation mapping targeting the illegal delivery class.
4. **Deterministic Rule Engine:** Enforcing format-specific ICC and MCC fielding restrictions, bouncer quotas, and Free Hit eligibility.


In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms

# Ensure project modules are importable
sys.path.append(os.path.abspath("."))
from src.cv_model import CricketClassifier, compute_gradcam
from src.rule_engine import CricketRuleEngine


## 1. Load Trained Neural Network Checkpoints

We initialize both model architectures and load the PyTorch checkpoints from `models/`.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load crease classifier (Law 21.5)
crease_model = CricketClassifier(num_classes=2).to(device)
crease_model.load_state_dict(torch.load("models/crease_classifier.pth", map_location=device))
crease_model.eval()

# Load waist classifier (Law 41.7)
waist_model = CricketClassifier(num_classes=2).to(device)
waist_model.load_state_dict(torch.load("models/waist_classifier.pth", map_location=device))
waist_model.eval()

params_count = sum(p.numel() for p in crease_model.parameters())
print(f"Device: {device}")
print(f"Loaded crease_classifier.pth and waist_classifier.pth")
print(f"Parameters per model: {params_count:,}")


## 2. Popping Crease Landing Evaluation (MCC Law 21.5)

Under MCC Law 21.5, the bowler's front foot must land with some part of the foot (whether grounded or in the air) behind the inside edge of the popping crease line.

We test two actual broadcast sample frames:
1. A fair delivery where the heel lands safely behind the crease line.
2. An overstep no-ball where the heel lands beyond the line.


In [ ]:
img_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def evaluate_crease_frame(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = img_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = crease_model(tensor)
        probs = F.softmax(logits, dim=1)[0]
    # classes: ['legal', 'no_ball'] -> index 1 is no_ball
    overstep_prob = probs[1].item()
    verdict = "NO-BALL (Overstep)" if overstep_prob > 0.50 else "LEGAL DELIVERY"
    return img, tensor, overstep_prob, verdict

# Evaluate sample frames
legal_img, legal_t, legal_prob, legal_verdict = evaluate_crease_frame("samples/crease/3_legal_delivery_behind_crease.jpg")
noball_img, noball_t, noball_prob, noball_verdict = evaluate_crease_frame("samples/crease/1_front_foot_noball_overstep.jpg")

print(f"Sample 1 (Legal Delivery):   Overstep Probability = {legal_prob*100:.2f}% -> {legal_verdict}")
print(f"Sample 2 (Overstep No-Ball): Overstep Probability = {noball_prob*100:.2f}% -> {noball_verdict}")


## 3. Spatial Interpretability via Grad-CAM

To verify that the model attends to the intersection between the bowler's foot and the popping crease rather than background noise, we compute the gradient-weighted class activation mapping (Grad-CAM) from the final convolutional stage (`stage4`).


In [ ]:
def get_gradcam_overlay(model, input_tensor, orig_img, target_idx=1):
    cam = compute_gradcam(model, input_tensor, target_idx)
    cam = np.maximum(cam, 0)
    if cam.max() > 1e-6:
        cam = cam / cam.max()
    
    # Resize activation map to original image resolution
    cam_resized = Image.fromarray((cam * 255).astype(np.uint8)).resize(orig_img.size, resample=Image.BICUBIC)
    return np.array(cam_resized)

cam_legal = get_gradcam_overlay(crease_model, legal_t, legal_img, target_idx=1)
cam_noball = get_gradcam_overlay(crease_model, noball_t, noball_img, target_idx=1)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].imshow(legal_img)
axes[0, 0].set_title(f"Legal Delivery: Original Frame")
axes[0, 0].axis("off")

axes[0, 1].imshow(legal_img)
axes[0, 1].imshow(cam_legal, cmap="jet", alpha=0.45)
axes[0, 1].set_title(f"Legal Delivery: Grad-CAM Overlay (Prob: {legal_prob*100:.1f}%)")
axes[0, 1].axis("off")

axes[1, 0].imshow(noball_img)
axes[1, 0].set_title(f"Overstep No-Ball: Original Frame")
axes[1, 0].axis("off")

axes[1, 1].imshow(noball_img)
axes[1, 1].imshow(cam_noball, cmap="jet", alpha=0.45)
axes[1, 1].set_title(f"Overstep No-Ball: Grad-CAM Overlay (Prob: {noball_prob*100:.1f}%)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()


## 4. Waist-High Full Toss Evaluation (MCC Law 41.7)

Under MCC Law 41.7, any delivery that passes or would have passed on the full above waist height of the striker standing upright at the popping crease is unfair and shall be called a no-ball.

Here we test side-on waist frames: a pitched delivery passing below waist level vs. a dangerous high full toss (beamer).


In [ ]:
def evaluate_waist_frame(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = img_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = waist_model(tensor)
        probs = F.softmax(logits, dim=1)[0]
    # classes: ['beamer', 'legal_waist'] -> index 0 is beamer
    beamer_prob = probs[0].item()
    verdict = "NO-BALL (Beamer)" if beamer_prob > 0.50 else "LEGAL HEIGHT"
    return img, beamer_prob, verdict

w_legal_img, w_legal_prob, w_legal_verdict = evaluate_waist_frame("samples/waist/1_waist_legal_height_delivery.jpg")
w_beamer_img, w_beamer_prob, w_beamer_verdict = evaluate_waist_frame("samples/waist/2_waist_high_beamer_noball.jpg")

print(f"Waist Sample 1 (Legal Height): Beamer Probability = {w_legal_prob*100:.2f}% -> {w_legal_verdict}")
print(f"Waist Sample 2 (High Beamer):  Beamer Probability = {w_beamer_prob*100:.2f}% -> {w_beamer_verdict}")


## 5. Deterministic ICC & MCC Rule Engine

In addition to visual inspection, deliveries must satisfy competition-specific rules:
- **30-Yard Circle Limit (ICC Clause 28.7):** Powerplay fielding caps (e.g., max 2 outside in T20 overs 1-6; max 5 in overs 7-20).
- **Leg-Side Limit (MCC Law 28.4):** At the instant of the bowler's delivery, there shall not be more than 5 fielders on the leg side.
- **Behind Square Leg Limit (MCC Law 28.4):** There shall not be more than 2 fielders behind the popping crease on the leg side.
- **Bouncer Quota (MCC Law 41.6):** Max 1 short-pitched delivery above shoulder height per over in T20I / 100-Ball; max 2 in IPL, ODI, and Test.


In [ ]:
rule_engine = CricketRuleEngine()

# Test Scenario A: T20 Over 4 (Powerplay), 3 fielders outside 30-yard circle (Violation)
viol_a, reasons_a = rule_engine.check_delivery(
    match_format="T20",
    over=3,
    ball_index_in_innings=21,
    bouncers_in_over_before=0,
    is_current_ball_bouncer=False,
    fielders_outside_circle=3,  # Max allowed in Powerplay is 2
    fielders_leg_side=3,
    fielders_behind_square=1,
)
print("Scenario A (Powerplay Circle Infringement):")
print(f"  Violation: {viol_a}, Reasons: {reasons_a}")

# Test Scenario B: IPL Over 18, 2nd bouncer bowled (Legal under IPL playing conditions, illegal under T20I)
viol_b_ipl, reasons_b_ipl = rule_engine.check_delivery(
    match_format="IPL",
    over=17,
    ball_index_in_innings=104,
    bouncers_in_over_before=1,
    is_current_ball_bouncer=True,
    fielders_outside_circle=5,
    fielders_leg_side=4,
    fielders_behind_square=1,
)
print("
Scenario B (IPL 2nd Bouncer Allowance):")
print(f"  IPL Violation: {viol_b_ipl} (IPL permits 2 bouncers per over)")

viol_b_t20, reasons_b_t20 = rule_engine.check_delivery(
    match_format="T20",
    over=17,
    ball_index_in_innings=104,
    bouncers_in_over_before=1,
    is_current_ball_bouncer=True,
    fielders_outside_circle=5,
    fielders_leg_side=4,
    fielders_behind_square=1,
)
print(f"  T20I Violation: {viol_b_t20}, Reasons: {reasons_b_t20}")


## 6. End-to-End Adjudication Function

This function integrates the dual convolutional neural networks with the rule engine to render a unified delivery decision.


In [ ]:
def adjudicate_delivery(crease_path, waist_path, match_context):
    reasons = []
    
    # 1. Crease camera check
    c_img = Image.open(crease_path).convert("RGB")
    c_t = img_transform(c_img).unsqueeze(0).to(device)
    with torch.no_grad():
        c_prob = F.softmax(crease_model(c_t), dim=1)[0][1].item()
    if c_prob > 0.50:
        reasons.append(f"Front-foot overstep detected ({c_prob*100:.1f}%) [MCC Law 21.5]")
        
    # 2. Waist camera check (if frame provided)
    w_prob = None
    if waist_path and os.path.exists(waist_path):
        w_img = Image.open(waist_path).convert("RGB")
        w_t = img_transform(w_img).unsqueeze(0).to(device)
        with torch.no_grad():
            w_prob = F.softmax(waist_model(w_t), dim=1)[0][0].item()
        if w_prob > 0.50:
            reasons.append(f"Waist-high full toss detected ({w_prob*100:.1f}%) [MCC Law 41.7]")
            
    # 3. Rule engine check
    rule_violation, rule_reasons = rule_engine.check_delivery(
        match_format=match_context.get("format", "T20"),
        over=match_context.get("over", 0),
        ball_index_in_innings=match_context.get("ball_index", 1),
        bouncers_in_over_before=match_context.get("bouncers_prior", 0),
        is_current_ball_bouncer=match_context.get("is_bouncer", False),
        fielders_leg_side=match_context.get("fielders_leg", 3),
        fielders_behind_square=match_context.get("fielders_behind_square", 1),
        fielders_outside_circle=match_context.get("fielders_outside_circle", 2),
    )
    if rule_violation:
        reasons.extend(rule_reasons)
        
    is_no_ball = len(reasons) > 0
    verdict = "NO-BALL" if is_no_ball else "LEGAL DELIVERY"
    free_hit = is_no_ball and match_context.get("format", "T20").upper() in ["T20", "IPL", "ODI", "100-BALL"]
    
    print("=" * 60)
    print(f"VERDICT: {verdict}")
    if free_hit:
        print("SIGNAL: FREE HIT AWARDED NEXT BALL")
    print(f"Crease Overstep Probability: {c_prob*100:.1f}%")
    if w_prob is not None:
        print(f"Waist Beamer Probability:    {w_prob*100:.1f}%")
    if reasons:
        print("Infringements Detected:")
        for r in reasons:
            print(f"  - {r}")
    else:
        print("All visual and fielding checks satisfied.")
    print("=" * 60)

# Run end-to-end check with real image frames and match context
adjudicate_delivery(
    crease_path="samples/crease/1_front_foot_noball_overstep.jpg",
    waist_path="samples/waist/1_waist_legal_height_delivery.jpg",
    match_context={
        "format": "T20",
        "over": 18,
        "ball_index": 110,
        "bouncers_prior": 0,
        "is_bouncer": False,
        "fielders_outside_circle": 5,
        "fielders_leg": 4,
        "fielders_behind_square": 1,
    }
)
